In [2]:
from pathlib import Path

# Find all CSV files in the processed MIMIC-III directory
data_dir = Path("../data/processed/mimic3")

print("Directory:", data_dir.resolve())
print("\nCSV files found:\n")

for file in sorted(data_dir.glob("*.csv")):
    print(file.name)

Directory: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/data/processed/mimic3

CSV files found:

clinical_events.csv
clinical_hourly_v1.csv
ecg_hrv_features.csv


In [3]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/mimic3/clinical_hourly_v1.csv"
)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Shape: (4556, 17)

Columns:
['subject_id', 'hadm_id', 'icustay_id', 'hour', 'gcs_eye', 'gcs_motor', 'gcs_verbal', 'heart_rate', 'map', 'resp_rate', 'spo2', 'gcs_total', 'previous_gcs', 'gcs_change', 'gcs_last_observed', 'previous_observed_gcs', 'sae']

First 5 rows:


,subject_id,hadm_id,icustay_id,hour,gcs_eye,gcs_motor,gcs_verbal,heart_rate,map,resp_rate,spo2,gcs_total,previous_gcs,gcs_change,gcs_last_observed,previous_observed_gcs,sae
0,10076,198503,201006,0,NaN,NaN,NaN,100.0,NaN,32.0,93.0,NaN,NaN,NaN,NaN,NaN,0
1,10076,198503,201006,1,NaN,NaN,NaN,104.5,NaN,33.5,94.0,NaN,NaN,NaN,NaN,NaN,0
2,10076,198503,201006,2,NaN,NaN,NaN,104.5,NaN,32.5,99.5,NaN,NaN,NaN,NaN,NaN,0
3,10076,198503,201006,3,4.0,6.0,5.0,103.0,NaN,36.0,99.0,15.0,NaN,NaN,15.0,NaN,0
4,10076,198503,201006,4,NaN,NaN,NaN,90.0,NaN,32.0,99.0,NaN,15.0,NaN,15.0,15.0,0


In [4]:
print("\nMissing values:")
display(df.isna().sum())

print("\nDtypes:")
print(df.dtypes)


Missing values:


subject_id                  0
hadm_id                     0
icustay_id                  0
hour                        0
gcs_eye                  3338
gcs_motor                3343
gcs_verbal               3342
heart_rate                 31
map                       921
resp_rate                 124
spo2                      203
gcs_total                3347
previous_gcs             3353
gcs_change               3385
gcs_last_observed          53
previous_observed_gcs      91
sae                         0
dtype: int64


Dtypes:
subject_id                 int64
hadm_id                    int64
icustay_id                 int64
hour                       int64
gcs_eye                  float64
gcs_motor                float64
gcs_verbal               float64
heart_rate               float64
map                      float64
resp_rate                float64
spo2                     float64
gcs_total                float64
previous_gcs             float64
gcs_change               float64
gcs_last_observed        float64
previous_observed_gcs    float64
sae                        int64
dtype: object


In [5]:
# ============================================================
# VERIFY SAE / DETERIORATION LABEL
# ============================================================

print("Total rows:", len(df))

print("\nSAE distribution:")
print(df["sae"].value_counts(dropna=False))

print("\nSAE percentage:")
print(df["sae"].value_counts(normalize=True) * 100)

print("\nICU stays:")
print(df["icustay_id"].nunique())

print("\nPatients:")
print(df["subject_id"].nunique())

print("\nICU stays containing SAE:")
print(
    df.groupby("icustay_id")["sae"]
      .max()
      .sum()
)

print("\nPatients containing SAE:")
print(
    df.groupby("subject_id")["sae"]
      .max()
      .sum()
)

Total rows: 4556

SAE distribution:
sae
0    4521
1      35
Name: count, dtype: int64

SAE percentage:
sae
0    99.231782
1     0.768218
Name: proportion, dtype: float64

ICU stays:
38

Patients:
25

ICU stays containing SAE:
16

Patients containing SAE:
12


In [6]:
# ============================================================
# SAE EVENT INSPECTION
# ============================================================

sae_events = df[df["sae"] == 1].copy()

print("SAE event rows:", len(sae_events))

display(
    sae_events[
        [
            "subject_id",
            "hadm_id",
            "icustay_id",
            "hour",
            "gcs_last_observed",
            "previous_observed_gcs",
            "gcs_change",
            "sae"
        ]
    ].sort_values(
        ["icustay_id", "hour"]
    ).head(30)
)

SAE event rows: 35


,subject_id,hadm_id,icustay_id,hour,gcs_last_observed,previous_observed_gcs,gcs_change,sae
11,10076,198503,201006,11,8.0,15.0,-7.0,1
16,10076,198503,201006,16,3.0,8.0,-5.0,1
169,10045,126949,203766,20,3.0,15.0,-12.0,1
289,10045,126949,203766,140,6.0,9.0,-3.0,1
290,10045,126949,203766,141,3.0,6.0,-3.0,1
320,41976,173269,205170,27,11.0,14.0,-3.0,1
356,41976,173269,205170,63,12.0,15.0,-3.0,1
428,41976,155297,209797,45,6.0,11.0,-5.0,1
697,10124,170883,222779,54,10.0,15.0,-5.0,1
761,10061,145203,223177,36,7.0,11.0,-4.0,1


In [7]:
# ============================================================
# CHECK RL STATE FEATURES
# ============================================================

state_columns = [
    "gcs_last_observed",
    "previous_observed_gcs",
    "gcs_change",
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "hour"
]

print("Missing values in proposed RL state:\n")

display(
    df[state_columns].isna().sum()
)

print("\nMissing percentage:\n")

display(
    (df[state_columns].isna().mean() * 100).round(2)
)

Missing values in proposed RL state:



gcs_last_observed          53
previous_observed_gcs      91
gcs_change               3385
heart_rate                 31
map                       921
resp_rate                 124
spo2                      203
hour                        0
dtype: int64


Missing percentage:



gcs_last_observed         1.16
previous_observed_gcs     2.00
gcs_change               74.30
heart_rate                0.68
map                      20.22
resp_rate                 2.72
spo2                      4.46
hour                      0.00
dtype: float64

In [8]:
# ============================================================
# CREATE RL-READY DATASET
# ============================================================

import pandas as pd
from pathlib import Path

rl_df = df.copy()

# ------------------------------------------------------------
# Sort chronologically within each ICU stay
# ------------------------------------------------------------

rl_df = rl_df.sort_values(
    ["icustay_id", "hour"]
).reset_index(drop=True)

# ------------------------------------------------------------
# RL state features
# ------------------------------------------------------------

state_columns = [
    "gcs_last_observed",
    "previous_observed_gcs",
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "hour"
]

# ------------------------------------------------------------
# Forward-fill clinical measurements within each ICU stay
# ------------------------------------------------------------

fill_columns = [
    "gcs_last_observed",
    "previous_observed_gcs",
    "heart_rate",
    "map",
    "resp_rate",
    "spo2"
]

rl_df[fill_columns] = (
    rl_df.groupby("icustay_id")[fill_columns]
         .ffill()
)

# ------------------------------------------------------------
# Median fallback for values still missing
# ------------------------------------------------------------

for col in fill_columns:
    rl_df[col] = rl_df[col].fillna(
        rl_df[col].median()
    )

# ------------------------------------------------------------
# Keep only required columns
# ------------------------------------------------------------

rl_columns = [
    "subject_id",
    "hadm_id",
    "icustay_id",
    "hour",
    *state_columns,
    "sae"
]

rl_df = rl_df[rl_columns].copy()

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("RL dataset shape:", rl_df.shape)

print("\nMissing values:")
print(rl_df.isna().sum())

print("\nSAE distribution:")
print(rl_df["sae"].value_counts())

print("\nICU stays:")
print(rl_df["icustay_id"].nunique())

print("\nPatients:")
print(rl_df["subject_id"].nunique())

RL dataset shape: (4556, 12)

Missing values:
subject_id               0
hadm_id                  0
icustay_id               0
hour                     0
gcs_last_observed        0
previous_observed_gcs    0
heart_rate               0
map                      0
resp_rate                0
spo2                     0
hour                     0
sae                      0
dtype: int64

SAE distribution:
sae
0    4521
1      35
Name: count, dtype: int64

ICU stays:
38

Patients:
25


In [9]:
# ============================================================
# SAVE RL DATASET
# ============================================================

output_path = Path(
    "../data/processed/mimic3/sae_rl_dataset.csv"
)

rl_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path.resolve())

Saved: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/data/processed/mimic3/sae_rl_dataset.csv
